# Simple RAG (Retrieval-Augmented Generation) System

## Overview

This project implements a foundational **Retrieval-Augmented Generation (RAG)** pipeline designed for processing and querying PDF documents. By encoding document content into a vector store, the system enables efficient semantic search and retrieval of relevant information in response to natural language queries.

## Key Components

1. **PDF Processing & Text Extraction** — Ingests PDF documents and extracts raw text content for downstream processing.
2. **Text Chunking** — Splits extracted content into manageable, semantically coherent segments to improve retrieval precision.
3. **Vector Store Creation** — Generates dense vector embeddings using [HuggingFace](https://huggingface.co/) models and indexes them with [FAISS](https://engineering.fb.com/2017/03/29/data-infrastructure/faiss-a-library-for-efficient-similarity-search/) for fast similarity search.
4. **Retriever Setup** — Configures a retrieval interface to query the vector store and surface the most relevant document chunks.
5. **System Evaluation** — Assesses retrieval quality and end-to-end RAG performance against defined benchmarks.

## Method Details

### Document Preprocessing

The pipeline begins by loading the target PDF using **PyPDFLoader**, which handles the extraction of raw text while preserving the document's page structure. Once loaded, the full text is passed through **RecursiveCharacterTextSplitter**, which divides the content into overlapping chunks of a configurable size. The overlap between chunks is intentional — it ensures that sentences or ideas spanning a chunk boundary are not lost, preserving contextual continuity across adjacent segments.

### Text Cleaning

Before the chunks are embedded, a custom cleaning function — `replace_t_with_space` — is applied to each one. PDFs are notoriously inconsistent in how they encode whitespace, and tab characters (`\t`) are a common artifact of the extraction process. This step normalizes the text so that the embedding model receives clean, well-formed input rather than fragmented or misaligned strings that could degrade retrieval quality.

### Vector Store Creation

Each cleaned text chunk is passed through **HugginFace's embedding model**, which transforms the raw text into a high-dimensional vector that captures its semantic meaning. These vectors are then indexed using **FAISS** (Facebook AI Similarity Search) — a library purpose-built for fast nearest-neighbor lookups in large vector spaces. The result is a compact, queryable store where similarity between a question and a document chunk can be computed in milliseconds, regardless of document size.

### Retriever Setup

On top of the FAISS index, a retriever is configured to return the **top 2 most semantically similar chunks** for any given query. Limiting retrieval to two chunks is a deliberate tradeoff — it keeps the context window concise and reduces noise when the retrieved content is passed downstream to a language model. This value is configurable and can be tuned based on the verbosity of the source document and the specificity of expected queries.

### Encoding Function

All of the above steps are consolidated into a single function: `encode_pdf`. This function accepts a file path and a set of chunking parameters, and returns a fully initialized retriever ready for querying. By encapsulating the entire preprocessing pipeline behind one interface, the system remains easy to integrate, test, and reuse across different documents without requiring changes to downstream components.

---

## Key Features

- **Modular Design** — The `encode_pdf` function acts as a self-contained pipeline. Loading, cleaning, embedding, and indexing are all handled internally, so the caller only needs to supply a file path and parameters.
- **Configurable Chunking** — Chunk size and overlap are exposed as parameters, making it straightforward to tune the system for documents with varying structure — from dense academic papers to sparse FAQ pages.
- **Efficient Similarity Search** — FAISS is optimized for high-dimensional vector spaces and scales well beyond what a brute-force cosine similarity search could handle, making it a practical choice even as document collections grow.
- **State-of-the-Art Embeddings** — HugginFace's embedding models are trained on large, diverse corpora, giving them strong generalization across domains. This means the system can meaningfully retrieve relevant content even when the query doesn't share exact phrasing with the source text.

---

## Usage Example

To illustrate retrieval in practice, the system is tested with the query:

> *"What is the main cause of climate change?"*

The retriever searches the vector store for the two chunks most semantically aligned with this question and returns them as context. This context can then be passed directly to a language model to generate a grounded, document-specific answer — which is the core value proposition of the RAG pattern.

---

## Evaluation

The system includes an `evaluate_rag` function to benchmark retrieval performance. While the specific metrics — such as precision, recall, or mean reciprocal rank — depend on the evaluation dataset provided, the function is designed to give a structured read on how well the retriever surfaces relevant content. This is especially important when tuning chunk size or the number of retrieved results, as small parameter changes can have a meaningful impact on downstream answer quality.

---

## Benefits of this Approach

- **Scalability** — Because the document is processed in chunks rather than as a single unit, the system can handle large files without hitting memory constraints or token limits. The FAISS index grows linearly with the number of chunks, not exponentially.
- **Flexibility** — Chunk size, overlap, and the number of retrieved results are all adjustable. This makes the system adaptable to a wide range of document types and query patterns without requiring structural changes to the code.
- **Efficiency** — FAISS performs approximate nearest-neighbor search using optimized indexing structures, delivering low-latency lookups even across hundreds of thousands of vectors — far faster than exhaustive search methods.
- **Semantic Understanding** — Unlike keyword-based retrieval, embedding-based search captures *meaning* rather than surface-level word matches. A query about "carbon emissions" will correctly surface passages about "greenhouse gases" or "fossil fuel combustion," even without lexical overlap.

---

## Conclusion

This RAG system is intentionally kept simple, but it reflects sound engineering principles: clean separation of concerns, configurable parameters, and a retrieval mechanism that scales. The combination of HugginFace embeddings and FAISS creates a strong semantic search layer that can be dropped in front of virtually any language model, transforming a static PDF into an interactive, queryable knowledge source.

For teams building document-grounded question-answering tools — whether for internal knowledge bases, legal document review, or technical support — this pipeline offers a practical and extensible starting point.

# Package Installation and Imports

The cell below installs all necessary packages required to run this notebook.

In [7]:
# Install the latest stable versions of all required packages
!pip install pypdf
!pip install PyMuPDF
!pip install python-dotenv
!pip install langchain-community
!pip install langchain_openai
!pip install rank_bm25
!pip install faiss-cpu
!pip install deepeval


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import os
from dotenv import load_dotenv
from getpass import getpass
from pathlib import Path
from typing import List, Union

In [9]:
def setup_api_keys():
    """Checks for required API keys and prompts the user if they are missing."""
    
    keys = {
        "OPENAI_API_KEY": "OpenAI",
        "HUGGINGFACEHUB_API_TOKEN": "Hugging Face"
    }
    
    for env_var, provider in keys.items():
        if not os.getenv(env_var):
            print(f"🔑 {provider} API Key not found.")
            # getpass masks the input so your key stays private
            os.environ[env_var] = getpass(f"Enter your {provider} API Key: ")
        else:
            print(f"✅ {provider} API Key is already set.")

In [10]:
# 1. Load existing environment variables from a .env file
load_dotenv()

True

In [11]:
# Execute setup
setup_api_keys()

✅ OpenAI API Key is already set.
✅ Hugging Face API Key is already set.


In [12]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# from helper_functions import (EmbeddingProvider,
#                               retrieve_context_per_question,
#                               replace_t_with_space,
#                               get_langchain_embedding_provider,
#                               show_context)

#from evaluation.evalute_rag import evaluate_rag

## Read Docs: The Universal Declaration of Human Rights (UDHR)

To validate the Simple RAG pipeline, we use the **Universal Declaration of Human Rights (UDHR)** as the primary test document. Its structure and content make it well-suited as a baseline benchmark for evaluating retrieval quality before moving on to more complex or domain-specific documents.

### Why This Document?

- **Clean Layout** — The UDHR follows a straightforward single-column structure, free of multi-column formatting, nested tables, or complex visual elements that commonly introduce parsing errors in loaders such as `pypdf`.
- **Fact-Dense, Numbered Structure** — The document is organized into 30 discrete, explicitly numbered articles. This makes it straightforward to verify whether the retriever is returning the correct chunk in response to a targeted query, rather than relying on subjective relevance judgments.
- **Optimal Scale** — At roughly 4–6 pages, the document is substantial enough to require chunking, yet compact enough to support rapid iteration and keep token costs low during development and testing.

### Reference Document

The official English-language version of the UDHR can be downloaded directly from the United Nations:

[Download the UDHR PDF — United Nations Official Document](https://www.un.org/en/about-us/universal-declaration-of-human-rights)

---

## Evaluation Benchmarks

The following questions are designed to assess the retrieval pipeline across three distinct capability dimensions. A well-functioning RAG implementation should answer each with high precision and minimal irrelevant context.

**1. Retrieval Accuracy**
> *"What does Article 17 say about property?"*

Tests whether the system can locate and return a specific, explicitly numbered section of the document — the most direct measure of retrieval precision.

**2. Semantic Understanding**
> *"Is everyone entitled to a fair trial?"*

Tests whether the retriever can surface relevant content when the query phrasing does not match the source text verbatim. Success here indicates that the embedding model is capturing meaning rather than relying on keyword overlap.

**3. Contextual Depth**
> *"What is the foundation of freedom, justice, and peace according to the preamble?"*

Tests retrieval from the document's introductory section rather than its main numbered articles. This verifies that the system indexes and retrieves all parts of the document uniformly, including prefatory content that may otherwise be underweighted.

In [13]:
from pathlib import Path

# Initialize the path object
folder = "data"
filename = "universal_declaration_of_human_rights_(udhr)"
path = Path(folder) / f"{filename}.pdf"

# Quick checks and metadata
if path.exists():
    print(f"File name: {path.name}")        # Just the filename
    print(f"Extension: {path.suffix}")      # .pdf
    print(f"Parent folder: {path.parent}")  # data/

File name: universal_declaration_of_human_rights_(udhr).pdf
Extension: .pdf
Parent folder: data


## Encode Document — PDF to Vector Store

Converts a PDF file into a searchable **FAISS vector store** for use in RAG pipelines.

**How it works:**

1. Loads PDF pages via `PyPDFLoader`
2. Splits text into overlapping chunks using `RecursiveCharacterTextSplitter`
3. Embeds chunks locally with `sentence-transformers/all-MiniLM-L6-v2` (CPU-friendly)
4. Indexes embeddings into a `FAISS` vector store for fast similarity search

> 💡 Switch `device` from `'cpu'` to `'cuda'` for GPU acceleration.

In [14]:
def clean_document_text(documents: List[Document]) -> List[Document]:
    """Replaces tab characters with spaces using modern string expansion."""
    for doc in documents:
        doc.page_content = doc.page_content.expandtabs(1)
    return documents

In [15]:
def encode_pdf(
    path: Union[str, Path], 
    chunk_size: int = 1000, 
    chunk_overlap: int = 200
) -> FAISS:
    """
    Encodes a PDF into a FAISS vector store using local HuggingFace embeddings.
    """
    # 1. Path Handling
    pdf_path = Path(path)
    if not pdf_path.exists():
        raise FileNotFoundError(f"No PDF found at: {pdf_path.absolute()}")

    # 2. Load PDF
    loader = PyPDFLoader(str(pdf_path))
    documents = loader.load()

    # 3. Clean and Split
    # We clean before splitting to ensure whitespace doesn't mess with chunk boundaries
    cleaned_docs = clean_document_text(documents)
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap,
        length_function=len,
        add_start_index=True 
    )
    texts = text_splitter.split_documents(cleaned_docs)

    # 4. Local Hugging Face Embeddings
    # This model is ~80MB and runs purely on your machine
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={'device': 'cpu'} 
    )

    # 5. Create and return Vector Store
    return FAISS.from_documents(texts, embeddings)

In [16]:
try:
    vector_db = encode_pdf(
        path=path, 
        chunk_size=1000, 
        chunk_overlap=200
    )
    print(f"Successfully encoded {path.name} into FAISS vector store.")
    
except FileNotFoundError as e:
    print(f"Error: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2712.78it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Successfully encoded universal_declaration_of_human_rights_(udhr).pdf into FAISS vector store.


In [17]:
# Quick check of the index size
total_chunks = vector_db.index.ntotal
print(f"Total chunks in vector store: {total_chunks}")

Total chunks in vector store: 16


## Create and Test Retriever - Querying the Vector Store

Turns a FAISS vector store into a **LangChain retriever** and fetches the most relevant chunks for a given query.

**How it works:**

1. Wraps `vector_db` as a retriever using cosine similarity search (`k=2` top chunks)
2. Invokes it with a natural language query via `.invoke()`
3. Prints a preview of each retrieved chunk with its source page number


> 💡 Increase `k` to retrieve more chunks — useful for longer, complex answers.

In [18]:
# 1. Create the retriever from your vector store
# 'k': 2 means it will return the top 2 most relevant text chunks
chunks_query_retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

# 2. Modern usage: Invoking the retriever
query = "What are the fundamental human rights mentioned?"
retrieved_docs = chunks_query_retriever.invoke(query)

# 3. Quick look at the results
for i, doc in enumerate(retrieved_docs):
    print(f"--- Chunk {i+1} (Page {doc.metadata.get('page', 'N/A')}) ---")
    print(f"{doc.page_content[:200]}...") # Print first 200 chars

--- Chunk 1 (Page 0) ---
Universal Declaration of Human Rights 
Preamble 
Whereas recognition of the inherent dignity and of the equal and inalienable 
rights of all members of the human family is the foundation of freedom, j...
--- Chunk 2 (Page 1) ---
teaching and education to promote respect for these rights and freedoms and by 
progressive measures, national and international, to secure their universal and 
effective recognition and observance, b...
